# BEIR + NanoBEIR Test Analysis (Relative)

This notebook combines:

- `examples/pooling/evaluation/results/nanobeir_test/nanobeir_pooling_sweep_20260212_083040`
- `examples/pooling/evaluation/results/beir_test/beir_pooling_sweep_20260212_095136`

Datasets are treated as separate variants, e.g. `scidocs` and `scidocs_nano`.
All relative plots use pool factor `1` as anchor (`100%`).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

## Parameters

In [ ]:
REPO_ROOT = Path.cwd().resolve().parents[2] if Path.cwd().name == "evaluation" else Path.cwd().resolve()
EVAL_DIR = REPO_ROOT / "examples" / "pooling" / "evaluation"

TARGET_MODEL = "mixedbread-ai/mxbai-edge-colbert-v0-32m"
KEY_METRICS = ["ndcg@10", "mrr@10", "map@100", "recall@10", "recall@100", "accuracy@10"]
PRIMARY_METRIC = "ndcg@10"
ANCHOR_POOL_FACTOR = 1

# Explicit source dirs for this analysis run.
SWEEP_SOURCES = [
    {
        "kind": "beir",
        "run_dir": EVAL_DIR / "results" / "beir_test" / "beir_pooling_sweep_20260212_095136",
        "dataset_suffix": "",
    },
    {
        "kind": "nanobeir",
        "run_dir": EVAL_DIR / "results" / "nanobeir_test" / "nanobeir_pooling_sweep_20260212_083040",
        "dataset_suffix": "_nano",
    },
]

# Filters
METHODS_TO_INCLUDE = ["span", "hierarchical", "kmeans", "kmeans_sk"]
POOL_FACTORS_TO_INCLUDE = None  # Example: [1, 2, 3]
DATASETS_TO_INCLUDE = None      # Example: ["scidocs", "scidocs_nano", "scifact", "scifact_nano"]
SAVE_RESULTS = False

assert PRIMARY_METRIC in KEY_METRICS
assert ANCHOR_POOL_FACTOR >= 1

In [ ]:
def canonical_method(pool_method: str | None, use_sklearn: bool | None) -> str:
    if pool_method == "kmeans" and bool(use_sklearn):
        return "kmeans_sk"
    return pool_method or "unknown"


def parse_nanobeir_scores(scores: dict[str, float]) -> dict[str, dict[str, float]]:
    dataset_results: dict[str, dict[str, float]] = {}
    for key, value in scores.items():
        if not key.startswith("Nano") or "_MaxSim_" not in key or "mean" in key.lower():
            continue
        left, metric = key.split("_MaxSim_", 1)
        dataset_name = left.replace("Nano", "", 1).lower()
        dataset_results.setdefault(dataset_name, {})[metric] = float(value)
    return dataset_results


def parse_beir_scores(scores: dict[str, dict[str, float]]) -> dict[str, dict[str, float]]:
    dataset_results: dict[str, dict[str, float]] = {}
    for dataset_name, metrics in scores.items():
        if not isinstance(metrics, dict):
            continue
        dataset_results[dataset_name.lower()] = {
            metric: float(value) for metric, value in metrics.items()
        }
    return dataset_results


def load_source(source: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    kind = source["kind"]
    run_dir = Path(source["run_dir"])
    suffix = source.get("dataset_suffix", "")

    if not run_dir.exists():
        raise FileNotFoundError(f"Missing run dir: {run_dir}")

    rows = []
    ds_rows = []

    pattern = "beir_*.json" if kind == "beir" else "nanobeir_*.json"
    for json_path in sorted(run_dir.glob(pattern)):
        if json_path.name == "overview_summary.json":
            continue

        payload = json.loads(json_path.read_text())
        if payload.get("model") != TARGET_MODEL:
            continue

        method = canonical_method(payload.get("pool_method"), payload.get("use_sklearn"))
        pool_factor = int(payload.get("pool_factor"))
        eval_time_s = float(payload.get("evaluation_time_seconds", np.nan))

        if kind == "beir":
            ds_results = parse_beir_scores(payload.get("scores", {}))
        else:
            ds_results = parse_nanobeir_scores(payload.get("scores", {}))

        rows.append({
            "source_kind": kind,
            "run_dir": run_dir.name,
            "results_json": json_path.name,
            "method": method,
            "pool_factor": pool_factor,
            "evaluation_time_seconds": eval_time_s,
        })

        for dataset_name, metrics in ds_results.items():
            dataset_label = f"{dataset_name}{suffix}"
            ds_rows.append({
                "source_kind": kind,
                "run_dir": run_dir.name,
                "results_json": json_path.name,
                "dataset": dataset_label,
                "base_dataset": dataset_name,
                "method": method,
                "pool_factor": pool_factor,
                **{metric: metrics.get(metric, np.nan) for metric in KEY_METRICS},
            })

    run_df = pd.DataFrame(rows)
    ds_df = pd.DataFrame(ds_rows)

    if not ds_df.empty:
        agg = (
            ds_df.groupby(["source_kind", "run_dir", "results_json", "method", "pool_factor"], as_index=False)
            [KEY_METRICS].mean()
            .rename(columns={m: f"mean_{m}" for m in KEY_METRICS})
        )
        run_df = run_df.drop(columns=["evaluation_time_seconds"], errors="ignore").merge(
            agg,
            on=["source_kind", "run_dir", "results_json", "method", "pool_factor"],
            how="left",
        )
        # Restore evaluation time (one row per json in rows)
        eval_time_map = pd.DataFrame(rows)[["results_json", "evaluation_time_seconds"]].drop_duplicates()
        run_df = run_df.merge(eval_time_map, on="results_json", how="left")

    return run_df, ds_df


def apply_filters(run_df: pd.DataFrame, ds_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out_run = run_df.copy()
    out_ds = ds_df.copy()

    out_run = out_run[out_run["method"].isin(METHODS_TO_INCLUDE)]
    out_ds = out_ds[out_ds["method"].isin(METHODS_TO_INCLUDE)]

    if POOL_FACTORS_TO_INCLUDE is not None:
        out_run = out_run[out_run["pool_factor"].isin(POOL_FACTORS_TO_INCLUDE)]
        out_ds = out_ds[out_ds["pool_factor"].isin(POOL_FACTORS_TO_INCLUDE)]

    if DATASETS_TO_INCLUDE is not None:
        keep = set(DATASETS_TO_INCLUDE)
        out_ds = out_ds[out_ds["dataset"].isin(keep)]
        valid_runs = out_ds[["results_json"]].drop_duplicates()
        out_run = out_run.merge(valid_runs, on="results_json", how="inner")

    out_run = out_run.sort_values(["method", "pool_factor", "source_kind"]).reset_index(drop=True)
    out_ds = out_ds.sort_values(["dataset", "method", "pool_factor"]).reset_index(drop=True)
    return out_run, out_ds


def add_relative_overall(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for metric in KEY_METRICS:
        mcol = f"mean_{metric}"
        rcol = f"rel_{metric}_pct"
        dcol = f"delta_{metric}_pct"

        anchors = (
            out[out["pool_factor"] == ANCHOR_POOL_FACTOR][["method", mcol]]
            .rename(columns={mcol: "anchor"})
            .drop_duplicates(["method"], keep="last")
        )
        out = out.merge(anchors, on="method", how="left")
        out[rcol] = 100.0 * out[mcol] / out["anchor"]
        out[dcol] = out[rcol] - 100.0
        out = out.drop(columns=["anchor"])
    return out


def add_relative_per_dataset(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for metric in KEY_METRICS:
        rcol = f"rel_{metric}_pct"
        dcol = f"delta_{metric}_pct"

        anchors = (
            out[out["pool_factor"] == ANCHOR_POOL_FACTOR][["dataset", "method", metric]]
            .rename(columns={metric: "anchor"})
            .drop_duplicates(["dataset", "method"], keep="last")
        )
        out = out.merge(anchors, on=["dataset", "method"], how="left")
        out[rcol] = 100.0 * out[metric] / out["anchor"]
        out[dcol] = out[rcol] - 100.0
        out = out.drop(columns=["anchor"])
    return out

In [ ]:
run_parts = []
ds_parts = []
for source in SWEEP_SOURCES:
    r, d = load_source(source)
    run_parts.append(r)
    ds_parts.append(d)

run_df_raw = pd.concat(run_parts, ignore_index=True)
ds_df_raw = pd.concat(ds_parts, ignore_index=True)

# Keep one row per method/pool/source config file.
run_df_raw = run_df_raw.drop_duplicates(["results_json"], keep="last").reset_index(drop=True)

datasets_available = sorted(ds_df_raw["dataset"].unique().tolist())
print("Available dataset labels:", datasets_available)

run_df, ds_df = apply_filters(run_df_raw, ds_df_raw)

if run_df.empty or ds_df.empty:
    raise ValueError("No rows left after filters. Check METHODS_TO_INCLUDE / POOL_FACTORS_TO_INCLUDE / DATASETS_TO_INCLUDE")

run_rel_df = add_relative_overall(run_df)
ds_rel_df = add_relative_per_dataset(ds_df)

print("Rows (overall):", len(run_rel_df))
print("Rows (per-dataset):", len(ds_rel_df))
print("Methods:", sorted(run_rel_df["method"].unique().tolist()))
print("Pool factors:", sorted(run_rel_df["pool_factor"].unique().tolist()))
print("Datasets:", sorted(ds_rel_df["dataset"].unique().tolist()))

In [ ]:
# Optional exports
if SAVE_RESULTS:
    out_dir = EVAL_DIR / "results"
    out_dir.mkdir(parents=True, exist_ok=True)

    run_rel_df.to_csv(out_dir / "analysis_test_overall_relative.csv", index=False)
    ds_rel_df.to_csv(out_dir / "analysis_test_per_dataset_relative.csv", index=False)

    print("Saved:")
    print(" -", out_dir / "analysis_test_overall_relative.csv")
    print(" -", out_dir / "analysis_test_per_dataset_relative.csv")

## Per-Dataset Relative Tables

In [ ]:
def plot_dataset_relative(dataset: str, metric: str = PRIMARY_METRIC, show_delta: bool = False):
    if metric not in KEY_METRICS:
        raise ValueError(f"Unknown metric: {metric}")

    rel_metric_col = f"rel_{metric}_pct"
    delta_metric_col = f"delta_{metric}_pct"
    y_col = delta_metric_col if show_delta else rel_metric_col

    part = ds_rel_df[ds_rel_df["dataset"] == dataset].sort_values(["method", "pool_factor"])
    if part.empty:
        raise ValueError(f"No rows for dataset {dataset}")

    plt.figure(figsize=(11, 6))
    for method in METHODS_TO_INCLUDE:
        mpart = part[part["method"] == method].sort_values("pool_factor")
        if mpart.empty:
            continue
        plt.plot(mpart["pool_factor"], mpart[y_col], marker="o", linewidth=2, label=method)

    baseline = 0.0 if show_delta else 100.0
    plt.axhline(baseline, color="black", linestyle="--", linewidth=1)
    title_suffix = "delta (% points)" if show_delta else "relative (%)"
    plt.title(f"{dataset}: {metric} {title_suffix}")
    plt.xlabel("Pool Factor")
    plt.ylabel(f"{metric} {title_suffix}")
    plt.xticks(sorted(part["pool_factor"].unique()))
    plt.legend(title="Method")
    plt.tight_layout()
    plt.show()

In [ ]:
# Example usage
plot_dataset_relative("scidocs", metric=PRIMARY_METRIC, show_delta=False)
plot_dataset_relative("scidocs_nano", metric=PRIMARY_METRIC, show_delta=False)

In [ ]:
rel_col = f"rel_{PRIMARY_METRIC}_pct"

# Small multiples for all datasets
datasets = sorted(ds_rel_df["dataset"].unique())
cols = 2
rows = int(np.ceil(len(datasets) / cols))

fig, axes = plt.subplots(rows, cols, figsize=(14, 4.2 * rows), sharex=True, sharey=False)
axes = np.array(axes).reshape(-1)

for ax, dataset in zip(axes, datasets):
    part = ds_rel_df[ds_rel_df["dataset"] == dataset].sort_values(["method", "pool_factor"])
    for method in METHODS_TO_INCLUDE:
        mpart = part[part["method"] == method].sort_values("pool_factor")
        if mpart.empty:
            continue
        ax.plot(mpart["pool_factor"], mpart[rel_col], marker="o", linewidth=1.8, label=method)
    ax.axhline(100.0, color="black", linestyle="--", linewidth=0.9)
    ax.set_title(dataset)
    ax.set_xlabel("Pool")
    ax.set_ylabel("Rel %")

for ax in axes[len(datasets):]:
    ax.axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=max(1, len(METHODS_TO_INCLUDE)), frameon=True)
fig.suptitle(f"Relative {PRIMARY_METRIC} by Dataset (pool={ANCHOR_POOL_FACTOR} => 100%)", y=1.02)
fig.tight_layout()
plt.show()